# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in an Anaconda prompt (PC) or Terminal (Mac), run:<br/>
            <code>conda env update --f environment.yml --prune</code><br/>
            Or if you used virtualenv rather than Anaconda, then run this from your activated environment in a Powershell (PC) or Terminal (Mac):<br/>
            <code>pip install -r requirements.txt</code>
            <br/>Then restart the kernel (Kernel menu >> Restart Kernel and Clear Outputs Of All Cells) to pick up the changes.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use GPT-4o and Claude-3.5-Sonnet, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please make the suggested switches to the models (3 cells down from here).
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic
from IPython.display import Markdown, display, update_display
import gradio as gr
import subprocess

In [2]:
# environment

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')

In [3]:
# initialize
# NOTE - option to use ultra-low cost models by uncommenting last 2 lines

openai = OpenAI()
claude = anthropic.Anthropic()
OPENAI_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-3-5-sonnet-20240620"

# Want to keep costs ultra-low? Uncomment these lines:
# OPENAI_MODEL = "gpt-4o-mini"
# CLAUDE_MODEL = "claude-3-haiku-20240307"

In [4]:
system_message = "You are an assistant that reimplements Python code in high performance R for an M1 Mac. "
system_message += "Respond only with R code; use comments sparingly and do not provide any explanation other than occasional comments. "
system_message += "The R response needs to produce an identical output in the fastest possible time."

In [5]:
def user_prompt_for(python):
    user_prompt = "Rewrite this Python code in R with the fastest possible implementation that produces identical output in the least time. "
    user_prompt += "Respond only with R code; do not explain your work other than a few comments. "
    user_prompt += "Pay attention to number types to ensure no int overflows. Remember to #include all necessary R packages.\n\n"
    user_prompt += python
    return user_prompt

In [6]:
def messages_for(python):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [7]:
# write to a file called optimized.R

def write_output(R):
    code = R.replace("```r","").replace("```","")
    with open("optimized.R", "w") as f:
        f.write(code)

In [8]:
# write to a file called optimized.cpp

# def write_output(cpp):
#     code = cpp.replace("```cpp","").replace("```","")
#     with open("optimized.cpp", "w") as f:
#         f.write(code)

In [9]:
def optimize_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        print(fragment, end='', flush=True)
    write_output(reply)

In [10]:
def optimize_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            print(text, end="", flush=True)
    write_output(reply)

In [11]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [12]:
exec(pi)

Result: 3.141592658589
Execution Time: 7.559624 seconds


In [13]:
optimize_gpt(pi)

```r
# No external R packages are needed

calculate <- function(iterations, param1, param2) {
  result <- 1.0
  for (i in seq(1, iterations)) {
    j1 <- i * param1 - param2
    j2 <- i * param1 + param2
    result <- result - (1/j1) + (1/j2)
  }
  return(result)
}

start_time <- Sys.time()
result <- calculate(100000000, 4, 1) * 4
end_time <- Sys.time()

cat(sprintf("Result: %.12f\n", result))
cat(sprintf("Execution Time: %.6f seconds\n", as.numeric(difftime(end_time, start_time, units = "secs"))))
```

In [14]:
exec(pi)

Result: 3.141592658589
Execution Time: 7.643621 seconds


# Executing the R Script

In [15]:
# Compile R and run the executable `optimized.R`

!Rscript optimized.R

Result: 3.141592658589
Execution Time: 5.753474 seconds


In [16]:
optimize_claude(pi)

```r
library(microbenchmark)

calculate <- function(iterations, param1, param2) {
  i <- 1:iterations
  j1 <- i * param1 - param2
  j2 <- i * param1 + param2
  result <- sum(1/j2 - 1/j1)
  return(result)
}

start_time <- Sys.time()
result <- calculate(100000000, 4, 1) * 4
end_time <- Sys.time()

cat(sprintf("Result: %.12f\n", result))
cat(sprintf("Execution Time: %.6f seconds\n", as.numeric(end_time - start_time, units="secs")))
```

In [17]:
!Rscript optimized.R

Result: -0.858407341903
Execution Time: 1.479338 seconds


In [18]:
python_hard = """# Be careful to support large number sizes

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [19]:
exec(python_hard)

Total Maximum Subarray Sum (20 runs): 10980
Execution Time: 24.284994 seconds


In [20]:
optimize_gpt(python_hard)

```r
# Load necessary package for handling large integers
library(RcppAlgos)

lcg <- function(seed, a=1664525, c=1013904223, m=2^32) {
  value <- seed
  while (TRUE) {
    value <- (a * value + c) %% m
    # Yield the current value
    yield(value)
  }
}

max_subarray_sum <- function(n, seed, min_val, max_val) {
  lcg_gen <- lcg(seed)
  random_numbers <- sapply(1:n, function(x) as.integer(nextElem(lcg_gen) %% (max_val - min_val + 1) + min_val))
  max_sum <- -Inf
  for (i in seq_len(n)) {
    current_sum <- 0
    for (j in i:n) {
      current_sum <- current_sum + random_numbers[j]
      max_sum <- max(max_sum, current_sum)
    }
  }
  return(max_sum)
}

total_max_subarray_sum <- function(n, initial_seed, min_val, max_val) {
  total_sum <- 0
  lcg_gen <- lcg(initial_seed)
  for (i in 1:20) {
    seed <- as.integer(nextElem(lcg_gen))
    total_sum <- total_sum + max_subarray_sum(n, seed, min_val, max_val)
  }
  return(total_sum)
}

# Parameters
n <- 10000
initial_seed <- 42
min_val <- -1

In [21]:
# Replace this with the right C++ compile + execute command for your platform

# !clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
# !./optimized

In [22]:
!Rscript optimized.R

Error in library(RcppAlgos) : there is no package called ‘RcppAlgos’
Execution halted


In [23]:
optimize_claude(python_hard)

#include Rcpp
#include RcppArmadillo

```r
library(Rcpp)
library(RcppArmadillo)

cppFunction('
uint64_t lcg(uint64_t seed, uint64_t a = 1664525, uint64_t c = 1013904223, uint64_t m = 4294967296) {
  return (a * seed + c) % m;
}
')

cppFunction('
double max_subarray_sum(int n, uint64_t seed, int min_val, int max_val) {
  std::vector<int> random_numbers(n);
  uint64_t value = seed;
  for (int i = 0; i < n; ++i) {
    value = lcg(value);
    random_numbers[i] = value % (max_val - min_val + 1) + min_val;
  }
  
  double max_sum = R_NegInf;
  double current_sum = 0;
  
  for (int i = 0; i < n; ++i) {
    current_sum = std::max(static_cast<double>(random_numbers[i]), current_sum + random_numbers[i]);
    max_sum = std::max(max_sum, current_sum);
  }
  
  return max_sum;
}
')

cppFunction('
double total_max_subarray_sum(int n, uint64_t initial_seed, int min_val, int max_val) {
  double total_sum = 0;
  uint64_t seed = initial_seed;
  
  for (int i = 0; i < 20; ++i) {
    seed = lcg(seed);
   

In [24]:
!Rscript optimized.R

using C++ compiler: ‘clang version 7.0.0 (tags/RELEASE_700/final)’
using SDK: ‘MacOSX15.4.sdk’
clang++ -arch arm64 -std=gnu++17 -I"/Library/Frameworks/R.framework/Resources/include" -DNDEBUG   -I"/Library/Frameworks/R.framework/Versions/4.4-arm64/Resources/library/Rcpp/include" -I"/private/var/folders/bs/zn7hxrnj0fx87gcf7r0jdfww0000gn/T/RtmpuFxEjz/sourceCpp-aarch64-apple-darwin20-1.0.14" -I/opt/R/arm64/include    -fPIC  -isysroot /Library/Developer/CommandLineTools/SDKs/MacOSX.sdk  -c file24822ba71627.cpp -o file24822ba71627.o
clang-7: warning: using sysroot for 'MacOSX' but targeting 'iPhone' [-Wincompatible-sysroot]
In file included from file24822ba71627.cpp:1:
In file included from /Library/Frameworks/R.framework/Versions/4.4-arm64/Resources/library/Rcpp/include/Rcpp.h:27:
In file included from /Library/Frameworks/R.framework/Versions/4.4-arm64/Resources/library/Rcpp/include/RcppCommon.h:30:
In file included from /Library/Frameworks/R.framework/Versions/4.4-arm64/Resources/library/R

In [25]:
def stream_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace('```r\n','').replace('```','')

In [26]:
def stream_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply.replace('```r\n','').replace('```','')

In [27]:
def optimize(python, model):
    if model=="GPT":
        result = stream_gpt(python)
    elif model=="Claude":
        result = stream_claude(python)
    else:
        raise ValueError("Unknown model")
    for stream_so_far in result:
        yield stream_so_far        

In [28]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=10, value=python_hard)
        R = gr.Textbox(label="R code:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
        convert = gr.Button("Convert code")

    convert.click(optimize, inputs=[python, model], outputs=[R])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [29]:
def execute_python(code):
    try:
        output = io.StringIO()
        sys.stdout = output
        exec(code)
    finally:
        sys.stdout = sys.__stdout__
    return output.getvalue()

In [30]:
# You'll need to change the code in the try block to compile the C++ code for your platform
# I pasted this into Claude's chat UI with a request for it to give me a version for an Intel PC,
# and it responded with something that looks perfect - you can try a similar approach for your platform.

# M1 Mac version to compile and execute optimized C++ code:

def execute_R(code):
        write_output(code)
        try:
            # compile_cmd = ["clang++", "-Ofast", "-std=c++17", "-march=armv8.5-a", "-mtune=apple-m1", "-mcpu=apple-m1", "-o", "optimized", "optimized.cpp"]
            # compile_result = subprocess.run(compile_cmd, check=True, text=True, capture_output=True)
            run_cmd = ["optimized.R"]
            run_result = subprocess.run(run_cmd, check=True, text=True, capture_output=True)
            return run_result.stdout
        except subprocess.CalledProcessError as e:
            return f"An error occurred:\n{e.stderr}"

In [31]:
css = """
.python {background-color: #306998;}
.R {background-color: #050;}
"""

In [ ]:
with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convert code from Python to R")
    with gr.Row():
        python = gr.Textbox(label="Python code:", value=python_hard, lines=10)
        R = gr.Textbox(label="R code:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
    with gr.Row():
        convert = gr.Button("Convert code")
    with gr.Row():
        python_run = gr.Button("Run Python")
        R_run = gr.Button("Run R")
    with gr.Row():
        python_out = gr.TextArea(label="Python result:", elem_classes=["python"])
        R_out = gr.TextArea(label="R result:", elem_classes=["cpp"])

    convert.click(optimize, inputs=[python, model], outputs=[R])
    python_run.click(execute_python, inputs=[python], outputs=[python_out])
    R_run.click(execute_R, inputs=[R], outputs=[R_out])

ui.launch(inbrowser=True, debug=True)

* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/woodzsan/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/woodzsan/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/woodzsan/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/blocks.py", line 2103, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/woodzsan/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/blocks.py", line 1650, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/woodzsan/anaconda3/envs/llms/lib/python3.11/site-packag